# **Prompting**

# Imports

In [1]:
import json
from pathlib import Path

import instructor
from anthropic import Anthropic
from dotenv import load_dotenv
from google.generativeai import GenerativeModel
from openai import OpenAI

from dreamai.ai import ModelName
from dreamai.prompt import BadExample, Example, Prompt

load_dotenv()

%load_ext autoreload
%autoreload 2
%reload_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Prompt

```python
class Prompt(BaseModel):
    id: str = Field(default_factory=lambda: str(uuid4()))
    system: SystemType = ""
    examples: list[Example | BadExample] = Field(default_factory=list)
    chat_history: list[MessageType] = Field(default_factory=list)
    version: float = DEFAULT_VERSION
    description: str = ""
    original_system: SystemType = ""
    change_history: list[ChangeRecord] = Field(default_factory=list)
    save_folder: Path | str = Path.cwd()

```

### Either a `system` or `examples` or `chat_history` must be provided

In [2]:
# financial_advisor_prompt = Prompt()

### Add a system prompt

In [3]:
save_folder = "financial_advisor_prompt"

system = """\
You are an AI financial advisor specializing in retirement planning.
Provide clear, concise, and accurate financial advice based on the user's questions.
"""

financial_advisor_prompt = Prompt(system=system, save_folder=save_folder)
financial_advisor_prompt.model_dump()

{'id': '69f7dd90-c515-43a7-a71b-99bb0637b6fd',
 'system': "You are an AI financial advisor specializing in retirement planning.\nProvide clear, concise, and accurate financial advice based on the user's questions.",
 'examples': [],
 'chat_history': [],
 'version': 1.0,
 'description': '',
 'original_system': "You are an AI financial advisor specializing in retirement planning.\nProvide clear, concise, and accurate financial advice based on the user's questions.",
 'change_history': [],
 'save_folder': 'financial_advisor_prompt'}

In [4]:
financial_advisor_prompt.messages

[{'role': 'system',
  'content': "You are an AI financial advisor specializing in retirement planning.\nProvide clear, concise, and accurate financial advice based on the user's questions."}]

### Add some examples

The `Example` class is used to make a "Good" example. Which is a user message and it's correct assistant response.

The `BadExample` class is used to make a "Bad" example. Which is a user message, it's incorrect assistant response, and the user feedback.


In [5]:
example_1 = Example(
    user="What's the difference between stocks and bonds?",
    assistant="Stocks represent ownership in a company, while bonds are loans to a company or government. Stocks generally offer higher potential returns but with higher risk, whereas bonds typically provide more stable, lower returns with less risk.",
)
example_1.messages

[{'role': 'user',
  'content': "What's the difference between stocks and bonds?"},
 {'role': 'assistant',
  'content': 'Stocks represent ownership in a company, while bonds are loans to a company or government. Stocks generally offer higher potential returns but with higher risk, whereas bonds typically provide more stable, lower returns with less risk.'}]

In [6]:
example_2 = Example(
    user="How should I start investing with a small budget?",
    assistant="To start investing with a small budget:\n1. Begin with a low-cost index fund or ETF\n2. Use a robo-advisor for automated, low-cost investing\n3. Set up regular, small contributions through dollar-cost averaging\n4. Consider micro-investing apps for very small amounts\n5. Prioritize tax-advantaged accounts like 401(k)s or IRAs",
)
example_2.messages

[{'role': 'user',
  'content': 'How should I start investing with a small budget?'},
 {'role': 'assistant',
  'content': 'To start investing with a small budget:\n1. Begin with a low-cost index fund or ETF\n2. Use a robo-advisor for automated, low-cost investing\n3. Set up regular, small contributions through dollar-cost averaging\n4. Consider micro-investing apps for very small amounts\n5. Prioritize tax-advantaged accounts like 401(k)s or IRAs'}]

In [7]:
example_3 = BadExample(
    user="Should I invest all my money in cryptocurrency?",
    assistant="Absolutely! Cryptocurrency is the future. You should definitely put all your savings into it for maximum returns.",
    feedback="This advice is irresponsible. Never recommend putting all savings into a single, highly volatile asset class. Instead, emphasize the importance of diversification and risk management.",
)
example_3.messages

[{'role': 'user',
  'content': 'Should I invest all my money in cryptocurrency?'},
 {'role': 'assistant',
  'content': 'Absolutely! Cryptocurrency is the future. You should definitely put all your savings into it for maximum returns.'},
 {'role': 'user',
  'content': 'This advice is irresponsible. Never recommend putting all savings into a single, highly volatile asset class. Instead, emphasize the importance of diversification and risk management.'}]

In [8]:
examples = [example_1, example_2, example_3]
financial_advisor_prompt.add_examples(examples=examples)

In [9]:
financial_advisor_prompt.messages

[{'role': 'system',
  'content': "You are an AI financial advisor specializing in retirement planning.\nProvide clear, concise, and accurate financial advice based on the user's questions."},
 {'role': 'user',
  'content': "What's the difference between stocks and bonds?"},
 {'role': 'assistant',
  'content': 'Stocks represent ownership in a company, while bonds are loans to a company or government. Stocks generally offer higher potential returns but with higher risk, whereas bonds typically provide more stable, lower returns with less risk.'},
 {'role': 'user',
  'content': 'How should I start investing with a small budget?'},
 {'role': 'assistant',
  'content': 'To start investing with a small budget:\n1. Begin with a low-cost index fund or ETF\n2. Use a robo-advisor for automated, low-cost investing\n3. Set up regular, small contributions through dollar-cost averaging\n4. Consider micro-investing apps for very small amounts\n5. Prioritize tax-advantaged accounts like 401(k)s or IRAs

### Add a new user message/query

In [10]:
query = "What's a good strategy for saving for retirement?"
financial_advisor_prompt.add_user_message(content=query)
financial_advisor_prompt.messages

[{'role': 'system',
  'content': "You are an AI financial advisor specializing in retirement planning.\nProvide clear, concise, and accurate financial advice based on the user's questions."},
 {'role': 'user',
  'content': "What's the difference between stocks and bonds?"},
 {'role': 'assistant',
  'content': 'Stocks represent ownership in a company, while bonds are loans to a company or government. Stocks generally offer higher potential returns but with higher risk, whereas bonds typically provide more stable, lower returns with less risk.'},
 {'role': 'user',
  'content': 'How should I start investing with a small budget?'},
 {'role': 'assistant',
  'content': 'To start investing with a small budget:\n1. Begin with a low-cost index fund or ETF\n2. Use a robo-advisor for automated, low-cost investing\n3. Set up regular, small contributions through dollar-cost averaging\n4. Consider micro-investing apps for very small amounts\n5. Prioritize tax-advantaged accounts like 401(k)s or IRAs

### Save

`save()` saves the prompt to a file named `f"{self.name}.json"` in the `self.save_folder`.

`save_folder` is set to the current working directory by default and can be changed when constructing the `Prompt` object.

`self.name` is `f"{self.id}_{self.version}"`

In [11]:
print(f"Save Folder: {financial_advisor_prompt.save_folder}")
print(f"Prompt Name: {financial_advisor_prompt.name}")

Save Folder: financial_advisor_prompt
Prompt Name: 69f7dd90-c515-43a7-a71b-99bb0637b6fd_1.0


In [12]:
save_path = financial_advisor_prompt.save()

### The JSON

In [13]:
saved_json = json.loads(Path(save_path).read_text())
saved_json

{'id': '69f7dd90-c515-43a7-a71b-99bb0637b6fd',
 'system': "You are an AI financial advisor specializing in retirement planning.\nProvide clear, concise, and accurate financial advice based on the user's questions.",
 'examples': [{'user': "What's the difference between stocks and bonds?",
   'assistant': 'Stocks represent ownership in a company, while bonds are loans to a company or government. Stocks generally offer higher potential returns but with higher risk, whereas bonds typically provide more stable, lower returns with less risk.'},
  {'user': 'How should I start investing with a small budget?',
   'assistant': 'To start investing with a small budget:\n1. Begin with a low-cost index fund or ETF\n2. Use a robo-advisor for automated, low-cost investing\n3. Set up regular, small contributions through dollar-cost averaging\n4. Consider micro-investing apps for very small amounts\n5. Prioritize tax-advantaged accounts like 401(k)s or IRAs'},
  {'user': 'Should I invest all my money i

### Load

`load()` receives a path to a saved prompt and returns a `Prompt` object.

It also accepts an `include_chat_history` argument to include the chat history in the loaded prompt. It defaults to `True`.

In [14]:
financial_advisor_prompt = Prompt.load(path=save_path)
financial_advisor_prompt.model_dump()

{'id': '69f7dd90-c515-43a7-a71b-99bb0637b6fd',
 'system': "You are an AI financial advisor specializing in retirement planning.\nProvide clear, concise, and accurate financial advice based on the user's questions.",
 'examples': [{'user': "What's the difference between stocks and bonds?",
   'assistant': 'Stocks represent ownership in a company, while bonds are loans to a company or government. Stocks generally offer higher potential returns but with higher risk, whereas bonds typically provide more stable, lower returns with less risk.'},
  {'user': 'How should I start investing with a small budget?',
   'assistant': 'To start investing with a small budget:\n1. Begin with a low-cost index fund or ETF\n2. Use a robo-advisor for automated, low-cost investing\n3. Set up regular, small contributions through dollar-cost averaging\n4. Consider micro-investing apps for very small amounts\n5. Prioritize tax-advantaged accounts like 401(k)s or IRAs'},
  {'user': 'Should I invest all my money i

In [15]:
financial_advisor_prompt.messages

[{'role': 'system',
  'content': "You are an AI financial advisor specializing in retirement planning.\nProvide clear, concise, and accurate financial advice based on the user's questions."},
 {'role': 'user',
  'content': "What's the difference between stocks and bonds?"},
 {'role': 'assistant',
  'content': 'Stocks represent ownership in a company, while bonds are loans to a company or government. Stocks generally offer higher potential returns but with higher risk, whereas bonds typically provide more stable, lower returns with less risk.'},
 {'role': 'user',
  'content': 'How should I start investing with a small budget?'},
 {'role': 'assistant',
  'content': 'To start investing with a small budget:\n1. Begin with a low-cost index fund or ETF\n2. Use a robo-advisor for automated, low-cost investing\n3. Set up regular, small contributions through dollar-cost averaging\n4. Consider micro-investing apps for very small amounts\n5. Prioritize tax-advantaged accounts like 401(k)s or IRAs

# AI

### List of models

```python
class ModelName(StrEnum):
    GPT = "gpt-4o-2024-08-06"
    GPT_MINI = "gpt-4o-mini"
    HAIKU = "claude-3-haiku-20240307"
    SONNET = "claude-3-5-sonnet-20240620"
    OPUS = "claude-3-opus-20240229"
    GEMINI_PRO = "gemini-1.5-pro-latest"
    GEMINI_PRO_EXP = "gemini-1.5-pro-exp-0801"
    GEMINI_FLASH = "gemini-1.5-flash-latest"
```

In [16]:
print(ModelName.GEMINI_FLASH)  # it will autocomplete

gemini-1.5-flash-latest


In [17]:
ask_gemini = instructor.from_gemini(GenerativeModel(model_name=ModelName.GEMINI_FLASH))
# ask_gpt = instructor.from_openai(OpenAI())
# ask_claude = instructor.from_anthropic(Anthropic())

### Kwargs

GPT, Claude, and Gemini all accept slightly different kwargs. The `Prompt` class will automatically convert the kwargs to the correct format for the model.

The `response_model` is the type of the response. It can be any type or pydantic model. Based on our prompt/query.

In [18]:
response_model = str
gemini_response = ask_gemini.create(
    response_model=str, **financial_advisor_prompt.gemini_kwargs
)
# gpt_response = ask_gpt.create(
#     model=ModelName.GPT_MINI, response_model=str, **financial_advisor_prompt.gpt_kwargs
# )
# claude_response = ask_claude.create(
#     model=ModelName.HAIKU, response_model=str, **financial_advisor_prompt.haiku_kwargs
# )

In [19]:
print(gemini_response)

A good retirement savings strategy involves diversifying investments, maximizing contributions to tax-advantaged accounts like 401(k)s and IRAs, and considering your risk tolerance and time horizon. It's also important to regularly review and adjust your portfolio as your financial situation and goals change.


In [20]:
financial_advisor_prompt.add_assistant_message(content=gemini_response)

In [21]:
financial_advisor_prompt.messages

[{'role': 'system',
  'content': "You are an AI financial advisor specializing in retirement planning.\nProvide clear, concise, and accurate financial advice based on the user's questions."},
 {'role': 'user',
  'content': "What's the difference between stocks and bonds?"},
 {'role': 'assistant',
  'content': 'Stocks represent ownership in a company, while bonds are loans to a company or government. Stocks generally offer higher potential returns but with higher risk, whereas bonds typically provide more stable, lower returns with less risk.'},
 {'role': 'user',
  'content': 'How should I start investing with a small budget?'},
 {'role': 'assistant',
  'content': 'To start investing with a small budget:\n1. Begin with a low-cost index fund or ETF\n2. Use a robo-advisor for automated, low-cost investing\n3. Set up regular, small contributions through dollar-cost averaging\n4. Consider micro-investing apps for very small amounts\n5. Prioritize tax-advantaged accounts like 401(k)s or IRAs

# Update system prompt

In [22]:
new_system = system + "\nGive me bullet points."
financial_advisor_prompt.update_system(new_system)

In [23]:
financial_advisor_prompt.name

'69f7dd90-c515-43a7-a71b-99bb0637b6fd_1.1'

In [24]:
financial_advisor_prompt.change_history[0].model_dump()

{'from_version': 1.0,
 'to_version': 1.1,
 'diff': "--- v1.0\n+++ v1.1\n@@ -1,2 +1,4 @@\n You are an AI financial advisor specializing in retirement planning.\n-Provide clear, concise, and accurate financial advice based on the user's questions.+Provide clear, concise, and accurate financial advice based on the user's questions.\n+\n+Give me bullet points.",
 'description': ''}

In [25]:
list(Path("financial_advisor_prompt").glob("*.json"))

[PosixPath('financial_advisor_prompt/69f7dd90-c515-43a7-a71b-99bb0637b6fd_1.1.json'),
 PosixPath('financial_advisor_prompt/69f7dd90-c515-43a7-a71b-99bb0637b6fd_1.0.json')]

In [26]:
financial_advisor_prompt.messages

[{'role': 'system',
  'content': "You are an AI financial advisor specializing in retirement planning.\nProvide clear, concise, and accurate financial advice based on the user's questions.\n\nGive me bullet points."},
 {'role': 'user',
  'content': "What's the difference between stocks and bonds?"},
 {'role': 'assistant',
  'content': 'Stocks represent ownership in a company, while bonds are loans to a company or government. Stocks generally offer higher potential returns but with higher risk, whereas bonds typically provide more stable, lower returns with less risk.'},
 {'role': 'user',
  'content': 'How should I start investing with a small budget?'},
 {'role': 'assistant',
  'content': 'To start investing with a small budget:\n1. Begin with a low-cost index fund or ETF\n2. Use a robo-advisor for automated, low-cost investing\n3. Set up regular, small contributions through dollar-cost averaging\n4. Consider micro-investing apps for very small amounts\n5. Prioritize tax-advantaged acc